# EDA — Multivariate Analysis
**Improvements:**
- Replaced deprecated `sns.distplot()` with `sns.histplot(kde=True)`
- Documented hard-coded outlier removal (737147) with IQR justification
- Fixed variable shadowing of `filtered_df`
- Removed unnecessary `.astype('category')` conversion

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re

pd.set_option('display.max_columns', None)

df = pd.read_csv('gurgaon_properties_cleaned_v2.csv').drop_duplicates()
print(f"Shape: {df.shape}")

## Property Type vs Price

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.barplot(data=df, x='property_type', y='price', estimator=np.median, ax=axes[0])
axes[0].set_title('Median Price by Property Type')
sns.boxplot(data=df, x='property_type', y='price', ax=axes[1])
axes[1].set_title('Price Distribution by Property Type')
plt.tight_layout(); plt.show()

## Property Type vs Built-Up Area
One extreme outlier (737147 sq.ft) found and removed using IQR-based check.

In [ ]:
# FIX: identify outlier statistically rather than hard-coding
Q1 = df['built_up_area'].quantile(0.25)
Q3 = df['built_up_area'].quantile(0.75)
IQR = Q3 - Q1
area_upper = Q3 + 3 * IQR   # 3×IQR for extreme outliers only
extreme_area = df[df['built_up_area'] > area_upper]
print(f"Extreme built_up_area outliers (>{area_upper:.0f} sqft): {len(extreme_area)}")
print(extreme_area[['property_type','sector','built_up_area','price']].to_string())

df = df[df['built_up_area'] <= area_upper].copy()
print(f"Shape after removal: {df.shape}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.barplot(data=df, x='property_type', y='built_up_area', estimator=np.median, ax=axes[0])
axes[0].set_title('Median Built-Up Area by Property Type')
sns.boxplot(data=df, x='property_type', y='built_up_area', ax=axes[1])
axes[1].set_title('Built-Up Area Distribution')
plt.tight_layout(); plt.show()

## Price per Sq.Ft Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
# FIX: replaced deprecated sns.distplot with sns.histplot(kde=True)
sns.histplot(df['price_per_sqft'].dropna(), kde=True, ax=axes[0])
axes[0].set_title('Price per Sq.Ft Distribution')
sns.boxplot(data=df, x='property_type', y='price_per_sqft', ax=axes[1])
axes[1].set_title('Price per Sq.Ft by Property Type')
plt.tight_layout(); plt.show()

print("Extreme price_per_sqft values (>100,000):")
print(df[df['price_per_sqft'] > 100000][['property_type','sector','price','price_per_sqft','area']].head(10))

## Bedroom & Bathroom Distributions

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.heatmap(pd.crosstab(df['property_type'], df['bedRoom']), annot=True, fmt='d', ax=axes[0])
axes[0].set_title('Property Type vs Bedrooms')
sns.barplot(data=df, x='bedRoom', y='price', estimator=np.median, ax=axes[1])
axes[1].set_title('Median Price by Bedroom Count')
plt.tight_layout(); plt.show()

## Age/Possession vs Price

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.heatmap(pd.crosstab(df['property_type'], df['agePossession']), annot=True, fmt='d', ax=axes[0])
axes[0].set_title('Count by Type & Age')
pivot = pd.pivot_table(df, index='property_type', columns='agePossession', values='price', aggfunc='mean')
sns.heatmap(pivot, annot=True, fmt='.2f', cmap='YlOrRd', ax=axes[1])
axes[1].set_title('Mean Price by Type & Age (Cr)')
plt.tight_layout(); plt.show()

## Furnishing vs Price

In [ ]:
# FIX: removed unnecessary .astype('category') — seaborn handles it automatically
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
pivot_furn = pd.pivot_table(df, index='property_type', columns='furnishing_type',
                             values='price', aggfunc='mean')
sns.heatmap(pivot_furn, annot=True, fmt='.2f', cmap='Blues', ax=axes[0])
axes[0].set_title('Mean Price by Type & Furnishing')
# FIX: no variable shadowing — use unique name for each filtered frame
area_furn_df = df[df['area'] < 10000].copy()
sns.scatterplot(data=area_furn_df, x='area', y='price', hue='furnishing_type', alpha=0.5, ax=axes[1])
axes[1].set_title('Area vs Price coloured by Furnishing')
plt.tight_layout(); plt.show()

## Sector Price Heatmap

In [ ]:
avg_price = df.groupby('sector')['price'].mean().reset_index()

def sector_sort_key(name):
    m = re.search(r'\d+', name)
    return int(m.group()) if m else float('inf')

avg_price['_key'] = avg_price['sector'].apply(sector_sort_key)
avg_price = avg_price.sort_values('_key').drop(columns='_key')

plt.figure(figsize=(5, 28))
sns.heatmap(avg_price.set_index('sector')[['price']], annot=True, fmt='.2f',
            cmap='YlOrRd', linewidths=0.5)
plt.title('Mean Price by Sector (Crores)')
plt.tight_layout(); plt.show()

## Correlation Heatmap

In [ ]:
numeric_df = df.select_dtypes(include='number')
plt.figure(figsize=(9, 7))
sns.heatmap(numeric_df.corr(), annot=True, cmap='coolwarm', fmt='.2f', vmin=-1, vmax=1)
plt.title('Correlation Matrix')
plt.tight_layout(); plt.show()

print("\nCorrelations with price (sorted):")
print(numeric_df.corr()['price'].sort_values(ascending=False))